In [17]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/combined_2024_bike_data.csv')
df.describe()

,대여일자,대여소번호,이용건수,운동량,탄소량,이동거리(M),이용시간(분)
count,1.233396e+06,1.233396e+06,1.233396e+06,1.232646e+06,1.232646e+06,1.233396e+06,1.233396e+06
mean,2.024066e+05,2.352501e+03,3.555189e+01,2.229309e+03,1.938465e+01,8.384110e+04,7.438958e+02
std,3.373785e+00,1.594367e+03,6.730375e+01,4.341666e+03,3.722986e+01,1.610801e+05,1.390664e+03
min,2.024010e+05,1.020000e+02,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.024040e+05,9.920000e+02,4.000000e+00,2.231800e+02,2.030000e+00,8.780000e+03,8.100000e+01
50%,2.024070e+05,2.037000e+03,1.200000e+01,7.846500e+02,7.050000e+00,3.048078e+04,2.740000e+02
75%,2.024090e+05,3.793000e+03,3.800000e+01,2.424747e+03,2.143000e+01,9.265555e+04,8.240000e+02
max,2.024120e+05,6.178000e+03,2.593000e+03,2.877765e+05,2.825580e+03,1.222876e+07,1.099940e+05


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1233396 entries, 0 to 1233395
Data columns (total 11 columns):
 #   Column   Non-Null Count    Dtype  
---  ------   --------------    -----  
 0   대여일자     1233396 non-null  int64  
 1   대여소번호    1233396 non-null  int64  
 2   대여소명     1233396 non-null  str    
 3   대여구분코드   1233396 non-null  str    
 4   성별       843095 non-null   str    
 5   연령대코드    1233396 non-null  str    
 6   이용건수     1233396 non-null  int64  
 7   운동량      1232646 non-null  float64
 8   탄소량      1232646 non-null  float64
 9   이동거리(M)  1233396 non-null  float64
 10  이용시간(분)  1233396 non-null  int64  
dtypes: float64(3), int64(4), str(4)
memory usage: 103.5 MB


In [24]:
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_ratio(%)": (df.isnull().mean() * 100).round(2),
}).sort_values("missing_count", ascending=False)

display(missing_summary[missing_summary["missing_count"] > 0])

,missing_count,missing_ratio(%)


In [20]:
# 결측치 처리 : fillna()
# 1. 성별 - 중앙값(median)
# df['성별'] = df['성별'].fillna( df['성별'].median() )

# # 2. Embarked - 최빈값(mode)
# df['Embarked'] = df['Embarked'].fillna( df['Embarked'].mode()[0] )
# 1. 성별 빈도수 계산 후 내림차순 정렬 (.sort_values 사용)

# 1. '성별' 컬럼의 결측치 개수 파악
null_count = df['성별'].isnull().sum()
print(f"성별 결측치 개수: {null_count}개")

# 2. 결측치 개수만큼 '남'과 '여'를 50:50 확률로 생성
# (데이터에 맞춰 '남'/'여' 대신 'M'/'F'로 변경하셔도 됩니다)
random_genders = np.random.choice(['M', 'F'], size=null_count, p=[0.5, 0.5])

# 3. 결측치 위치에 생성한 랜덤 성별 집어넣기
df.loc[df['성별'].isnull(), '성별'] = random_genders

# 4. 결과 확인 (남녀 비율이 비슷해졌는지 확인)
df['성별'].value_counts()

성별 결측치 개수: 0개


성별
M    640731
F    592475
m       120
f        70
Name: count, dtype: int64

In [22]:
# 성별 컬럼의 소문자를 모두 대문자로 변경 (공백 제거 포함)
df['성별'] = df['성별'].str.strip().str.upper()

# 결과 확인
df['성별'].value_counts()

성별
M    640851
F    592545
Name: count, dtype: int64

In [23]:
# 2. 운동량, 탄소량- 최빈값(mode)
df['운동량'] = df['운동량'].fillna( df['운동량'].mode()[0] )
df['탄소량'] = df['탄소량'].fillna( df['탄소량'].mode()[0] )

In [25]:
# 1. '대' 문자열을 지우고 숫자만 남기기
df['연령대코드'] = df['연령대코드'].str.replace('대', '', regex=False)

# 2. 숫자로 강제 변환 (결측치나 '미지정' 글자가 섞여있다면 에러 방지를 위해 errors='coerce' 사용)
df['연령대코드'] = pd.to_numeric(df['연령대코드'], errors='coerce')

# 3. 변환 결과 확인
df['연령대코드'].value_counts()

연령대코드
20.0    194174
30.0    190885
40.0    181443
50.0    164137
60.0    118159
Name: count, dtype: int64

In [26]:
df.describe()

,대여일자,대여소번호,연령대코드,이용건수,운동량,탄소량,이동거리(M),이용시간(분)
count,1.233396e+06,1.233396e+06,848798.000000,1.233396e+06,1.233396e+06,1.233396e+06,1.233396e+06,1.233396e+06
mean,2.024066e+05,2.352501e+03,37.893751,3.555189e+01,2.227954e+03,1.937286e+01,8.384110e+04,7.438958e+02
std,3.373785e+00,1.594367e+03,13.585976,6.730375e+01,4.340693e+03,3.722160e+01,1.610801e+05,1.390664e+03
min,2.024010e+05,1.020000e+02,20.000000,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.024040e+05,9.920000e+02,30.000000,4.000000e+00,2.225600e+02,2.030000e+00,8.780000e+03,8.100000e+01
50%,2.024070e+05,2.037000e+03,40.000000,1.200000e+01,7.836300e+02,7.040000e+00,3.048078e+04,2.740000e+02
75%,2.024090e+05,3.793000e+03,50.000000,3.800000e+01,2.422977e+03,2.141000e+01,9.265555e+04,8.240000e+02
max,2.024120e+05,6.178000e+03,60.000000,2.593000e+03,2.877765e+05,2.825580e+03,1.222876e+07,1.099940e+05


In [28]:
df['연령대코드'].isnull().sum()

np.int64(384598)